In [ ]:
!pip install -q --upgrade pip

In [ ]:
!pip install -U google-adk chromadb groq langchain tiktoken 

In [ ]:
!pip show chromadb

In [ ]:
!pip install --upgrade langchain langchain-chroma langchain-groq chromadb langchain-huggingface

In [ ]:
!pip install langchain -U langchain_community

In [ ]:
#To get dataset from website
!wget -q https://www.dropbox.com/s/vs6ocyvpzzncvwh/new_articles.zip

In [ ]:
#To unzip the documents
!unzip -q new_articles.zip -d new_articles

In [ ]:
import os
os.environ["GROQ_API_KEY"]="Paste your Groq API key here"

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.document_loaders import DirectoryLoader, TextLoader

In [ ]:
loader = DirectoryLoader("/content/new_articles", glob="./*.txt", loader_cls=TextLoader)

In [ ]:
document=loader.load()

In [ ]:
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the splitter (syntax unchanged)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# If 'document' is a list of Document objects (as returned by loaders)
texts = text_splitter.split_documents(document)

# If 'document' is just a long text string, use this instead:
# texts = text_splitter.split_text(document)

In [ ]:
texts[0].page_content

In [ ]:
texts[1].page_content

In [ ]:
len(texts)

# Creating DB

In [ ]:
from langchain import embeddings
persist_directory='db'
embedding=HuggingFaceEmbeddings()
vectordb=Chroma.from_documents(documents=texts,embedding=embedding,persist_directory=persist_directory)

In [ ]:
# Persists the db to disk
# vectordb.persist() # This line is not needed in this version of Chroma
vectordb=None

In [ ]:
vectordb=Chroma(persist_directory=persist_directory, embedding_function=embedding)

# Make a retriever

In [ ]:
retriever=vectordb.as_retriever()

In [ ]:
docs=retriever.invoke("How much money did Microsoft raised ?")

In [ ]:
len(docs)

In [ ]:
retriever=vectordb.as_retriever(search_kwargs={"k":2})

In [ ]:
retriever.search_kwargs

In [ ]:
docs2=retriever.invoke("How much money did Microsoft raised ?")

In [ ]:
len(docs2)

In [ ]:
docs2

# Make a chain

In [ ]:
!pip install --upgrade langchain

In [ ]:
from langchain_groq import ChatGroq
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate # Use ChatPromptTemplate

llm=ChatGroq(model="llama-3.1-8b-instant", temperature=0)
#Create the chain to answer questions
#Step 2: Define your system prompt
system_prompt = """
You are a knowledgeable AI assistant. Use the provided context to answer the question precisely.
If the answer is not contained in the context, say "I don't know" rather than guessing.
Context: {context}
"""
prompt = ChatPromptTemplate.from_template(system_prompt + "\n\nQuestion: {question}\nAnswer:")

#Step 3: Create the document-combining chain (stuff = simplest type)
qa_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

#Step 4: Create the retrieval chain
retrieval_chain = create_retrieval_chain(
    retriever=retriever,  # your retriever from Chroma / FAISS / Pinecone etc.
    combine_documents_chain=qa_chain,
)

In [ ]:
def process_llm_response(llm_response):
  print(llm_response['result'])
  print('\n\nSources:')
  for source in llm_response["source_documents"]:
      print(source.metadata['source'])

In [ ]:
query="How much money did Microsoft raise?"

In [ ]:
llm_response=qa_chain(query)

In [ ]:
process_llm_response(llm_response)

# Delete the DB

In [ ]:
!zip -r dib.zip ./db

In [ ]:
#To cleanup, you can delete the collection
vectordb.delete_collection()

In [ ]:
#Delete the directory
!rm -rf db/